In [ ]:
import pandas as pd
from pathlib import Path

relative_path = Path("checkpoints/note02_data_quality.pkl")

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Không tìm thấy file: {relative_path}"
    )
df_path = project_root / relative_path

note02_data_quality = project_root / "checkpoints/note02_data_quality.pkl"

note02_state = pd.read_pickle(note02_data_quality)

df = note02_state["df"]
generation_sources = note02_state["generation_sources"]
system_sources = note02_state["system_sources"]

print("Loaded:", note02_data_quality)
print("Dataset shape:", df.shape)
print("Year range:", df["Year"].min(), "→", df["Year"].max())
print("Areas:", df["Area"].nunique())
print("Electricity sources:", df["Electricity source"].nunique())

Loaded: d:\phan tich data\Ember_Electricity\checkpoints\note02_data_quality.pkl
Dataset shape: (104203, 24)
Year range: 1985 → 2025
Areas: 224
Electricity sources: 17


In [3]:
source_summary = (
    df.groupby("Electricity source")
      .agg(
          rows=("Area", "size"),
          areas=("Area", "nunique"),
          first_year=("Year", "min"),
          last_year=("Year", "max"),
          total_generation_twh=("Generation (TWh)", "sum")
      )
      .sort_values("total_generation_twh", ascending=False)
)

source_summary

,rows,areas,first_year,last_year,total_generation_twh
Electricity source,,,,,
Demand,6305,224,1985,2025,3015911.029
Total generation,6305,224,1985,2025,3015720.290
Fossil,6305,224,1985,2025,1918039.145
Clean,6298,224,1985,2025,1097681.144
Coal,6189,220,1985,2025,1095687.037
Renewables,6298,224,1985,2025,691576.895
Gas,6122,217,1985,2025,673938.374
"Hydro, bioenergy and other renewables",6298,224,1985,2025,510945.810
Hydro,6190,222,1985,2025,447245.344


In [4]:
global_trend = (
    df[
        (df["Area"] == "World")
        & (df["Electricity source"].isin([
            "Total generation",
            "Renewables",
            "Fossil"
        ]))
    ][
        [
            "Year",
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)"
        ]
    ]
    .sort_values(["Year", "Electricity source"])
)

global_trend

,Year,Electricity source,Generation (TWh),Share of generation (%)
102525,2000,Fossil,9917.323,64.712
102523,2000,Renewables,2867.514,18.711
102526,2000,Total generation,15325.294,100.000
102541,2001,Fossil,10129.432,65.151
102539,2001,Renewables,2804.967,18.041
...,...,...,...,...
102907,2024,Renewables,9805.497,31.641
102910,2024,Total generation,30990.182,100.000
102925,2025,Fossil,18357.373,57.691
102923,2025,Renewables,10653.411,33.480


In [5]:
global_sources = (
    df[
        (df["Area"] == "World")
        & (df["Electricity source"].isin([
            "Solar",
            "Wind",
            "Hydro",
            "Coal",
            "Gas"
        ]))
    ][
        [
            "Year",
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)"
        ]
    ]
    .sort_values(["Year", "Electricity source"])
)

global_sources

,Year,Electricity source,Generation (TWh),Share of generation (%)
102518,2000,Coal,5841.078,38.114
102519,2000,Gas,2747.126,17.925
102514,2000,Hydro,2632.570,17.178
102512,2000,Solar,1.092,0.007
102513,2000,Wind,31.150,0.203
...,...,...,...,...
102918,2025,Coal,10610.915,33.347
102919,2025,Gas,6924.779,21.762
102914,2025,Hydro,4435.956,13.941
102912,2025,Solar,2780.204,8.737


In [6]:
country_core_coverage = (
    df[
        (df["Area type"] == "Country or economy")
        & (df["Electricity source"].isin([
            "Total generation",
            "Renewables",
            "Fossil",
            "Solar",
            "Wind"
        ]))
    ]
    .groupby("Year")
    .agg(
        countries=("Area", "nunique"),
        generation_nonnull=("Generation (TWh)", "count"),
        share_nonnull=("Share of generation (%)", "count")
    )
    .sort_index()
)

country_core_coverage.tail(15)

,countries,generation_nonnull,share_nonnull
Year,,,
2011,208,1031,1031
2012,209,1037,1037
2013,209,1038,1038
2014,209,1040,1040
2015,209,1040,1040
2016,209,1040,1040
2017,209,1041,1041
2018,209,1041,1041
2019,209,1041,1041


In [7]:
country_panel = (
    df[
        (df["Area type"] == "Country or economy")
        & (df["Year"].between(2012, 2022))
        & (df["Electricity source"] == "Total generation")
    ]
    .groupby("Area")["Year"]
    .nunique()
    .sort_values()
)

country_panel.describe()

count    209.0
mean      11.0
std        0.0
min       11.0
25%       11.0
50%       11.0
75%       11.0
max       11.0
Name: Year, dtype: float64

In [8]:
country_source_panel = (
    df[
        (df["Area type"] == "Country or economy")
        & (df["Year"].between(2012, 2022))
        & (df["Electricity source"].isin([
            "Total generation",
            "Renewables",
            "Fossil",
            "Solar",
            "Wind"
        ]))
    ]
    .groupby(["Electricity source", "Area"])["Year"]
    .nunique()
    .groupby(level=0)
    .agg(["count", "min", "max", "mean"])
)

country_source_panel

,count,min,max,mean
Electricity source,,,,
Fossil,209,11,11,11.000000
Renewables,209,11,11,11.000000
Solar,209,11,11,11.000000
Total generation,209,11,11,11.000000
Wind,206,3,11,10.912621


In [9]:
wind_coverage = (
    df[
        (df["Area type"] == "Country or economy")
        & (df["Year"].between(2012, 2022))
        & (df["Electricity source"] == "Wind")
    ]
    .groupby("Area")["Year"]
    .nunique()
    .sort_values()
)

wind_coverage[wind_coverage < 11]

Area
El Salvador         3
Montenegro          6
Kosovo              9
North Macedonia     9
Slovenia           10
Name: Year, dtype: int64

In [10]:
country_generation_2022 = (
    df[
        (df["Area type"] == "Country or economy")
        & (df["Year"] == 2022)
        & (df["Electricity source"] == "Total generation")
    ][
        ["Area", "Generation (TWh)", "Share of generation (%)"]
    ]
    .sort_values("Generation (TWh)", ascending=False)
)

country_generation_2022.describe()

,Generation (TWh),Share of generation (%)
count,209.000000,209.0
mean,138.557981,100.0
std,697.131809,0.0
min,0.010000,100.0
25%,1.040000,100.0
50%,11.330000,100.0
75%,54.740000,100.0
max,8800.417000,100.0


In [11]:
country_generation_2022.head(10)

,Area,Generation (TWh),Share of generation (%)
19286,China,8800.417,100.0
99310,United States,4286.890,100.0
44252,India,1803.289,100.0
78500,Russia,1166.940,100.0
48435,Japan,1040.570,100.0
13238,Brazil,677.160,100.0
16945,Canada,664.080,100.0
87340,South Korea,622.590,100.0
36913,Germany,566.880,100.0
33454,France,468.370,100.0


In [12]:
country_generation_2022.tail(10)

,Area,Generation (TWh),Share of generation (%)
81468,Sao Tome and Principe,0.09,100.0
40872,Guinea-Bissau,0.08,100.0
100636,Vanuatu,0.08,100.0
94261,Tonga,0.08,100.0
80212,Saint Pierre and Miquelon,0.05,100.0
21424,Cook Islands (the),0.04,100.0
65762,Nauru,0.04,100.0
50336,Kiribati,0.04,100.0
31431,Falkland Islands (Malvinas),0.01,100.0
63620,Montserrat,0.01,100.0


In [13]:
country_mix_2022 = (
    df[
        (df["Area type"] == "Country or economy")
        & (df["Year"] == 2022)
        & (df["Electricity source"].isin([
            "Renewables",
            "Fossil",
            "Solar",
            "Wind"
        ]))
    ][
        ["Area", "Electricity source", "Generation (TWh)", "Share of generation (%)"]
    ]
    .sort_values(["Area", "Electricity source"])
)

country_mix_2022.head(20)

,Area,Electricity source,Generation (TWh),Share of generation (%)
1220,Afghanistan,Fossil,0.13,13.542
1218,Afghanistan,Renewables,0.83,86.458
1206,Afghanistan,Solar,0.09,9.375
1207,Afghanistan,Wind,0.00,0.000
2061,Albania,Fossil,0.00,0.000
2059,Albania,Renewables,6.97,100.000
2047,Albania,Solar,0.16,2.296
2048,Albania,Wind,0.00,0.000
2486,Algeria,Fossil,90.61,99.060
2484,Algeria,Renewables,0.86,0.940


In [14]:
country_mix_check = (
    country_mix_2022
    .pivot(
        index="Area",
        columns="Electricity source",
        values="Share of generation (%)"
    )
)

country_mix_check.describe()

Electricity source,Fossil,Renewables,Solar,Wind
count,209.000000,209.000000,209.000000,206.000000
mean,59.777249,36.716761,4.514057,4.065267
std,32.968458,31.979382,6.705845,7.934451
min,0.000000,0.000000,0.000000,0.000000
25%,28.421000,9.218000,0.245000,0.000000
50%,66.667000,27.359000,2.128000,0.082500
75%,88.889000,63.089000,5.879000,4.359250
max,100.000000,100.000000,50.000000,54.096000


In [15]:
mix_balance = (
    country_mix_check["Fossil"]
    + country_mix_check["Renewables"]
)

mix_balance.describe()

count    209.000000
mean      96.494010
std       10.803374
min       37.073000
25%      100.000000
50%      100.000000
75%      100.000000
max      100.000000
dtype: float64

In [16]:
mix_balance[mix_balance < 99.9].sort_values().head(15)

Area
France         37.073
Slovakia       40.015
Ukraine        44.238
Belgium        53.482
Hungary        55.814
Slovenia       58.009
Switzerland    60.872
Czechia        62.953
Finland        64.888
Bulgaria       67.264
Sweden         69.985
South Korea    71.723
Armenia        73.380
Spain          79.596
Romania        79.740
dtype: float64

In [17]:
nuclear_share_2022 = (
    df[
        (df["Area type"] == "Country or economy")
        & (df["Year"] == 2022)
        & (df["Electricity source"] == "Nuclear")
    ][
        ["Area", "Share of generation (%)"]
    ]
    .set_index("Area")
    .loc[mix_balance[mix_balance < 99.9].sort_values().head(15).index]
)

nuclear_share_2022

,Share of generation (%)
Area,
France,62.927
Slovakia,59.985
Ukraine,55.762
Belgium,46.518
Hungary,44.187
Slovenia,41.991
Switzerland,39.128
Czechia,37.048
Finland,35.112


In [18]:
# ============================================================
# RQ2 – Cơ cấu sản xuất điện khác nhau như thế nào giữa các quốc gia?
# Sub-question: Quốc gia nào có tỷ trọng Nuclear cao nhất năm 2022?
# ============================================================

nuclear_2022 = (
    df[
        (df["Area type"] == "Country or economy")
        & (df["Year"] == 2022)
        & (df["Electricity source"] == "Nuclear")
    ][
        ["Area", "Generation (TWh)", "Share of generation (%)"]
    ]
    .sort_values("Share of generation (%)", ascending=False)
)

nuclear_2022.head(10)

,Area,Generation (TWh),Share of generation (%)
33444,France,294.73,62.927
84821,Slovakia,15.92,59.985
97655,Ukraine,62.18,55.762
10074,Belgium,43.88,46.518
43156,Hungary,15.81,44.187
85420,Slovenia,5.61,41.991
90399,Switzerland,24.15,39.128
24516,Czechia,31.02,37.048
32832,Finland,25.34,35.112
14258,Bulgaria,16.46,32.737


In [19]:
# ============================================================
# NOTE 3 CHECKPOINT
# RQ1 + RQ2 – EDA tổng quan và cơ cấu điện
# ============================================================

note03_eda_overview = (
    project_root / "checkpoints/note03_eda_overview.pkl"
)

note03_state = {
    "df": df,
    "generation_sources": generation_sources,
    "system_sources": system_sources,
    "source_summary": source_summary,
    "global_trend": global_trend,
    "global_sources": global_sources,
    "country_core_coverage": country_core_coverage,
    "country_panel": country_panel,
    "country_source_panel": country_source_panel,
    "wind_coverage": wind_coverage,
    "country_generation_2022": country_generation_2022,
    "country_mix_2022": country_mix_2022,
    "country_mix_check": country_mix_check,
    "mix_balance": mix_balance,
    "nuclear_2022": nuclear_2022
}

pd.to_pickle(note03_state, note03_eda_overview)

print("Saved:", note03_eda_overview)
print("Note 3 checkpoint created successfully.")

Saved: d:\phan tich data\Ember_Electricity\checkpoints\note03_eda_overview.pkl
Note 3 checkpoint created successfully.
